In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 7.5 Text Mining in Higher Ed – Sentiment Analysis
- VADER lexicon-based sentiment (no training needed)
- Validating VADER against labeled ground truth
- Crosstab: where VADER agrees/disagrees

## Setup

In [ ]:
import random
import pandas as pd

ML_Survey_Data = pd.read_csv('../data/ML_Survey_Data.csv')
display(ML_Survey_Data)

## VADER Sentiment
Compound score: ≥0.05 positive, ≤-0.05 negative, else neutral. Fast baseline, no labels needed.

In [ ]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
df_Sentiment = ML_Survey_Data[['SID', 'Free_Response_Text']].copy()
df_Sentiment['vader_compound'] = df_Sentiment['Free_Response_Text'].apply(lambda t: sia.polarity_scores(t)['compound'])

def vader_label(score, pos_thresh=0.05, neg_thresh=-0.05):
    if score >= pos_thresh:
        return 'positive'
    if score <= neg_thresh:
        return 'negative'
    return 'neutral'

df_Sentiment['vader_label'] = df_Sentiment['vader_compound'].apply(vader_label)
df_Sentiment['vader_label'].value_counts()

## Ground-Truth Validation
Generate synthetic labeled comments tied to HS_GPA, then compare against VADER's labels with a crosstab.

In [ ]:
df_training = pd.read_csv('../data/training.csv')

positive_bank = [
    'I really enjoyed this course and felt supported by my instructors.',
    'The assignments were challenging in a good way, and I learned a lot.',
]
neutral_bank = [
    'The semester was okay overall, with some ups and downs.',
    'Some parts of the course were useful, and others were less clear.',
]
negative_bank = [
    'I struggled a lot and often felt like I did not know where to get help.',
    'The workload was overwhelming and I felt stressed most weeks.',
]

def make_sentiment_comment(gpa_norm):
    if gpa_norm > 3.7:
        return random.choice(positive_bank), 'positive'
    elif gpa_norm < 3.4:
        return random.choice(negative_bank), 'negative'
    label = random.choice(['neutral', 'positive', 'negative'])
    bank = {'neutral': neutral_bank, 'positive': positive_bank, 'negative': negative_bank}[label]
    return random.choice(bank), label

rows = [(sid, *make_sentiment_comment(float(g))) for sid, g in zip(df_training['SID'], df_training['HS_GPA'])]
sent_df = pd.DataFrame(rows, columns=['SID', 'Comment', 'Labeled Sentiment'])
sent_df1 = pd.merge(sent_df, df_Sentiment[['SID', 'vader_label']], on='SID', how='left')
sent_df1['Labeled Sentiment'].value_counts()

In [ ]:
ct = pd.crosstab(sent_df1['Labeled Sentiment'], sent_df1['vader_label'])
ct

## Summary
- VADER is a fast baseline that needs no labels — but the crosstab shows it doesn't always agree with ground truth, especially on nuanced or domain-specific language.
- Always pair sentiment with topics (7.4) and real example comments — never report a sentiment score alone.

**Next:** 7.6 fuses all of this (structured + text) into one model-ready matrix.